In [4]:
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw/swat")        # use the root-based version from before if the cwd differs

def load_swat(path):
    df = pd.read_csv(path, low_memory=False)
    if str(df.columns[0]).startswith("Unnamed"):     # stage row used as header, so re-read with the real one
        df = pd.read_csv(path, header=1, low_memory=False)
    df.columns = df.columns.str.strip()
    return df

normal = load_swat(RAW / "SWaT_Dataset_Normal_v1.csv")
attack = load_swat(RAW / "SWaT_Dataset_Attack_v0.csv")   # use your actual file name

for name, df in [("normal", normal), ("attack", attack)]:
    print(name, df.shape)
print(normal.columns.tolist())
print(normal.head(3))
print(attack["Normal/Attack"].value_counts())
print(normal["Normal/Attack"].value_counts())

normal (495000, 53)
attack (449919, 53)
['Timestamp', 'FIT101', 'LIT101', 'MV101', 'P101', 'P102', 'AIT201', 'AIT202', 'AIT203', 'FIT201', 'MV201', 'P201', 'P202', 'P203', 'P204', 'P205', 'P206', 'DPIT301', 'FIT301', 'LIT301', 'MV301', 'MV302', 'MV303', 'MV304', 'P301', 'P302', 'AIT401', 'AIT402', 'FIT401', 'LIT401', 'P401', 'P402', 'P403', 'P404', 'UV401', 'AIT501', 'AIT502', 'AIT503', 'AIT504', 'FIT501', 'FIT502', 'FIT503', 'FIT504', 'P501', 'P502', 'PIT501', 'PIT502', 'PIT503', 'FIT601', 'P601', 'P602', 'P603', 'Normal/Attack']
                Timestamp  FIT101    LIT101  MV101  P101  P102    AIT201  \
0   22/12/2015 4:30:00 PM     0.0  124.3135      1     1     1  251.9226   
1   22/12/2015 4:30:01 PM     0.0  124.3920      1     1     1  251.9226   
2   22/12/2015 4:30:02 PM     0.0  124.4705      1     1     1  251.9226   

     AIT202    AIT203  FIT201  ...  P501  P502    PIT501  PIT502  PIT503  \
0  8.313446  312.7916     0.0  ...     1     1  9.100231     0.0  3.3485   
1  8.3

In [ ]:
for df in (normal, attack):
    df["Timestamp"] = pd.to_datetime(df["Timestamp"].str.strip(), dayfirst=True)

# The label column has typos in some SWaT releases (e.g. "A ttack"), so normalize it
attack["label"] = (attack["Normal/Attack"].str.replace(" ", "").str.strip()
                   .str.lower().eq("attack").astype(int))
normal["label"] = (normal["Normal/Attack"].str.replace(" ", "").str.strip()
                   .str.lower().eq("attack").astype(int))

for name, df in [("normal", normal), ("attack", attack)]:
    step = df["Timestamp"].diff().dropna()
    print(name, df["Timestamp"].min(), "->", df["Timestamp"].max())
    print("  steps != 1s:", (step != pd.Timedelta(seconds=1)).sum(),
          "| dup timestamps:", df["Timestamp"].duplicated().sum(),
          "| NaNs:", df.isna().sum().sum(),
          "| attack rows:", df["label"].sum())

In [ ]:
seg_id = (attack["label"].diff().fillna(0) != 0).cumsum()
segs = (attack[attack["label"] == 1]
        .groupby(seg_id)["Timestamp"].agg(start="min", end="max", rows="count"))
segs["duration_s"] = (segs["end"] - segs["start"]).dt.total_seconds() + 1
print(len(segs), "attack segments")
segs

In [ ]:
feature_cols = [c for c in normal.columns if c not in ("Timestamp", "Normal/Attack", "label")]
print(len(feature_cols), "features (expect 51)")

stage = pd.Series({c: next((ch for ch in c if ch.isdigit()), "?") for c in feature_cols})
print(stage.value_counts().sort_index())      # P1..P6 sizes (the stage digit is the first digit in the name)

nunique = normal[feature_cols].nunique()
print("Constant columns:", nunique[nunique == 1].index.tolist())
print("Likely actuators (<=5 unique values):", nunique[nunique <= 5].index.tolist())